# Download antiobiotics data from chEMBL

In [2]:
import os
from chembl_webresource_client.new_client import new_client
import pandas as pd
import time
from tqdm import tqdm

/users/sghosh6/.conda/envs/pubchemData/lib/python3.13/site-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


In [8]:
# Base directory path
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'

# Construct full paths for subdirectories 
Bacillus_subtilis_dataDir = os.path.join(dataDir, 'Bacillus_subtilis/')
Bacillus_cereus_dataDir = os.path.join(dataDir, 'Bacillus_cereus/')
modelBuilding_dataDir = os.path.join(dataDir, 'modelBuildingData/')

### working with manually downloaded data from chEMBL

In [1]:
import os
import zipfile
import pandas as pd
import glob
import numpy as np
import warnings
from pathlib import Path
import csv
import matplotlib.pyplot as plt
import re

In [3]:
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
modelBuildingDataDir = os.path.join(dataDir, 'modelBuildingData')

### The code below will process the data based on the bacteria name user provided

In [4]:
import pandas as pd
import numpy as np
import os
import csv
import re
import matplotlib.pyplot as plt
from io import StringIO

def readChemblCsvFile(filePath):
    """
    Read ChEMBL CSV file with optimized settings for large files.
    """
    print(f"Reading ChEMBL CSV file: {filePath}")
    if not os.path.exists(filePath):
        print(f"Error: File not found at {filePath}")
        return None
    
    try:
        # Define dtypes upfront to reduce memory and increase speed
        dtype_dict = {
            'Standard Value': 'float32',
            'pChEMBL Value': 'float32',
            'Molecular Weight': 'float32',
            'AlogP': 'float32',
            'Standard Type': 'category',
            'Standard Units': 'category',
            'Standard Relation': 'category',
            'Target Organism': 'category'
        }
        
        dataFrame = pd.read_csv(
            filePath,
            sep=';',
            quotechar='"',
            quoting=csv.QUOTE_ALL,
            doublequote=True,
            skipinitialspace=True,
            on_bad_lines='skip',
            encoding='utf-8',
            dtype=dtype_dict,
            engine='c',  # Faster C engine
            na_values=['', 'NA', 'N/A', 'null']
        )
        return dataFrame
    except Exception as e:
        print(f"Error reading file: {e}")
        return None


def processBacteriaData(bacteriaName):
    """
    Optimized main function to process bioactivity and compound data.
    """
    
    if not os.path.exists(bacteriaDataDir):
        print(f"Error: Data directory for '{bacteriaName}' not found at {bacteriaDataDir}")
        return

    # ### ChEMBL Bioassay Data
    print("\n--- Processing Bioassay Data ---")
    bioactivityFile = os.path.join(bacteriaDataDir, f'{bacteriaName}_bioactivities.csv')
    chemblData = readChemblCsvFile(bioactivityFile)

    if chemblData is None:
        return

    print(f"Dataset loaded successfully with Shape: {chemblData.shape}")

    # #### Keep only relevant properties - avoid .copy() if not modifying
    selectedColumns = ['Smiles', 'Standard Value', 'Standard Units', 'Standard Type',
                      'Standard Relation', 'pChEMBL Value', 'Molecule ChEMBL ID',
                      'Molecular Weight', 'AlogP', 'Target ChEMBL ID',
                      'Target Name', 'Target Organism']

    chemblDataCleaned = chemblData[[col for col in selectedColumns if col in chemblData.columns]]
    print(f"Dataset shape after selecting columns: {chemblData.shape} → {chemblDataCleaned.shape}")
    
    outputPath = os.path.join(modelBuildingDataDir, f'{bacteriaName}Bioassay_chEMBL_cleaned.csv')
    chemblDataCleaned.to_csv(outputPath, index=False)
    print(f"Cleaned bioassay data saved to: {outputPath}")

    # ### Count Standard Type and Generate Plot
    print("\n--- Analyzing Standard Types ---")
    typeCounts = chemblDataCleaned["Standard Type"].value_counts()
    total = typeCounts.sum()
    legendLabels = [f"{stype} ({count}, {count/total:.1%})" for stype, count in typeCounts.items()]

    plt.figure(figsize=(10, 8))
    wedges, texts, autotexts = plt.pie(
        typeCounts, labels=typeCounts.index, autopct='%1.1f%%',
        startangle=140, wedgeprops={'edgecolor': 'black'}, textprops={'fontsize': 9}
    )
    plt.legend(wedges, legendLabels, title="Standard Type", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))
    plt.title(f"Distribution of Standard Types for {bacteriaName}")
    plt.tight_layout()
    plotPath = os.path.join(modelBuildingDataDir, f'{bacteriaName}_standard_types.png')
    plt.savefig(plotPath, dpi=150)
    print(f"Standard types plot saved to: {plotPath}")
    plt.close()  # Close to free memory

    # ### Check for empty (NaN) values
    print("\n--- Checking for NaN values ---")
    nanCounts = chemblDataCleaned.isnull().sum()
    totalRows = len(chemblDataCleaned)
    for column, nanCount in nanCounts.items():
        percentage = (nanCount / totalRows) * 100
        print(f"{column:25s}: {nanCount:6,} ({percentage:5.1f}%)")

    # ### Count unique compounds
    if "Molecule ChEMBL ID" in chemblDataCleaned.columns:
        uniqueCount = chemblDataCleaned["Molecule ChEMBL ID"].nunique()
        print(f"\nNumber of unique compounds for {bacteriaName}: {uniqueCount}")

    # ### ChEMBL compounds data - OPTIMIZED VERSION
    print("\n--- Processing Compounds Data ---")
    compoundsFile = os.path.join(bacteriaDataDir, f'{bacteriaName}_compounds.csv')
    if not os.path.exists(compoundsFile):
        print(f"Error: Compounds file not found at {compoundsFile}")
        return

    # Try using pandas directly with error handling
    try:
        chemblCompoundData = pd.read_csv(
            compoundsFile,
            sep=';',
            quotechar='"',
            encoding='utf-8',
            on_bad_lines='skip',
            engine='c',
            low_memory=False
        )
        print("Successfully read compounds file with pandas")
    except Exception as e:
        print(f"Pandas failed, using optimized manual parser: {e}")
        chemblCompoundData = readCompoundsFileOptimized(compoundsFile)
    
    # Standardize column names efficiently
    chemblCompoundData.columns = chemblCompoundData.columns.str.replace(r"[^\w]+", "_", regex=True).str.strip("_").str.lower()
    
    # Standardize the ChEMBL ID column name for merging
    if 'molecule_chembl_id' in chemblCompoundData.columns:
        chemblCompoundData.rename(columns={'molecule_chembl_id': 'chembl_id'}, inplace=True)

    # Convert numeric columns efficiently
    numCols = ["molecular_weight", "alogp", "polar_surface_area", "hba", "hbd", 
               "ro5_violations", "rotatable_bonds", "qed_weighted", "heavy_atoms"]
    
    for c in numCols:
        if c in chemblCompoundData.columns:
            chemblCompoundData[c] = pd.to_numeric(chemblCompoundData[c], errors="coerce").astype('float32')
    
    print(f"Compounds data shape: {chemblCompoundData.shape}")
    outputPath = os.path.join(modelBuildingDataDir, f'{bacteriaName}Compounds_chEMBL_cleaned.csv')
    chemblCompoundData.to_csv(outputPath, index=False)
    print(f"Cleaned compounds data saved to: {outputPath}")

    # ### Combine Bioassay and Compound Data
    print("\n--- Combining Bioassay and Compound Data ---")
    colsToImport = ['chembl_id', 'type', 'polar_surface_area', 'hba', 'hbd', 'rotatable_bonds', 
                    'qed_weighted', 'heavy_atoms', 'aromatic_rings', 'ro5_violations', 
                    'molecular_formula', 'inchi_key', 'inchi']
    cmpdSub = chemblCompoundData[[c for c in colsToImport if c in chemblCompoundData.columns]]

    # Use merge with indicator to check merge quality
    chemblDataCombined = chemblDataCleaned.merge(
        cmpdSub, 
        how='left', 
        left_on='Molecule ChEMBL ID', 
        right_on='chembl_id',
        suffixes=('', '_cmp'),
        validate='m:1'  # Many-to-one merge validation
    )
    
    if 'chembl_id' in chemblDataCombined.columns:
        chemblDataCombined.drop(columns=['chembl_id'], inplace=True)

    # ### Rename columns efficiently
    rename_dict = {
        "Smiles": "smiles", 
        "Standard Value": "standard_value", 
        "Standard Units": "standard_units",
        "Standard Type": "standard_type", 
        "Standard Relation": "standard_relation", 
        "pChEMBL Value": "pchembl_value",
        "Molecule ChEMBL ID": "molecule_chembl_id", 
        "Molecular Weight": "molecular_weight",
    }
    chemblDataCombined.rename(columns=rename_dict, inplace=True)

    # ### Reorder columns
    desiredOrder = [
        "molecule_chembl_id", "smiles", "molecular_formula", "inchi_key", "inchi", "molecular_weight", "type",
        "standard_value", "standard_units", "standard_relation", "standard_type", "pchembl_value", "AlogP",
        "polar_surface_area", "hba", "hbd", "rotatable_bonds", "qed_weighted", "heavy_atoms", "aromatic_rings",
        "ro5_violations", "Target ChEMBL ID", "Target Name", "Target Organism"
    ]
    
    existing = [c for c in desiredOrder if c in chemblDataCombined.columns]
    extras = [c for c in chemblDataCombined.columns if c not in existing]
    chemblDataCombined = chemblDataCombined[existing + extras]

    outputPath = os.path.join(modelBuildingDataDir, f'{bacteriaName}Data_chEMBL_combined.csv')
    chemblDataCombined.to_csv(outputPath, index=False)
    print(f"Final combined data saved to: {outputPath} with shape {chemblDataCombined.shape}")
    print("\nProcessing complete.")
    
    return chemblDataCombined


def readCompoundsFileOptimized(filePath):
    """
    Optimized fallback parser for problematic CSV files.
    Uses list comprehension and minimal string operations.
    """
    with open(filePath, "r", encoding="utf-8", errors="replace") as f:
        # Read all at once - faster than line by line
        content = f.read()
    
    # Split into lines
    lines = content.split('\n')
    
    # Use csv module which is implemented in C and much faster
    import csv
    from io import StringIO
    
    reader = csv.reader(StringIO(content), delimiter=';', quotechar='"')
    rows = list(reader)
    
    if not rows:
        return pd.DataFrame()
    
    # Create DataFrame directly
    df = pd.DataFrame(rows[1:], columns=rows[0])
    return df




### Processing data files for `Bacillus_subtilis` bacteria 

In [5]:
bacteriaName = 'Bacillus_subtilis'  
bacteriaDataDir = os.path.join(dataDir, 'chEMBL/' + bacteriaName)
processBacteriaData(bacteriaName)


--- Processing Bioassay Data ---
Reading ChEMBL CSV file: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/chEMBL/Bacillus_subtilis/Bacillus_subtilis_bioactivities.csv


/tmp/ipykernel_370924/4160898751.py:31: DtypeWarning: Columns (31,45) have mixed types. Specify dtype option on import or set low_memory=False.
  dataFrame = pd.read_csv(


Dataset loaded successfully with Shape: (34560, 48)
Dataset shape after selecting columns: (34560, 48) → (34560, 12)
Cleaned bioassay data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Bacillus_subtilisBioassay_chEMBL_cleaned.csv

--- Analyzing Standard Types ---
Standard types plot saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Bacillus_subtilis_standard_types.png

--- Checking for NaN values ---
Smiles                   :    172 (  0.5%)
Standard Value           :  4,018 ( 11.6%)
Standard Units           :  5,020 ( 14.5%)
Standard Type            :      0 (  0.0%)
Standard Relation        :  4,034 ( 11.7%)
pChEMBL Value            : 34,360 ( 99.4%)
Molecule ChEMBL ID       :      0 (  0.0%)
Molecular Weight         :    172 (  0.5%)
AlogP                    :  2,275 (  6.6%)
Target ChEMBL ID         :      0 (  0.0%)
Target Name              :      0 (  0.0%)
Target Organism          :     

,molecule_chembl_id,smiles,molecular_formula,inchi_key,inchi,molecular_weight,type,standard_value,standard_units,standard_relation,...,hba,hbd,rotatable_bonds,qed_weighted,heavy_atoms,aromatic_rings,ro5_violations,Target ChEMBL ID,Target Name,Target Organism
0,CHEMBL221435,CC1=C[C@H]2[C@@H](C(C)(C)O)CC[C@@](C)(O)[C@H]2CC1,C15H26O2,XOUUSQWJTXEKIT-PWNZVWSESA-N,InChI=1S/C15H26O2/c1-10-5-6-13-11(9-10)12(14(2...,238.369995,Small molecule,NaN,NaN,NaN,...,2.0,2.0,1.0,0.69,17.0,0.0,0.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
1,CHEMBL8,O=C(O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O,C17H18FN3O3,MYSWGUAQZAJSOK-UHFFFAOYSA-N,InChI=1S/C17H18FN3O3/c18-13-7-11-14(8-15(13)20...,331.350006,Small molecule,0.25,ug.mL-1,'=',...,5.0,2.0,3.0,0.89,24.0,2.0,0.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
2,CHEMBL606360,COc1ccc(C(=O)/C=C/c2cn(-c3ccccc3)nc2-c2ccc(Cl)...,C26H21ClN2O3,KEINEPLOMWOUNE-XNTDXEJSSA-N,InChI=1S/C26H21ClN2O3/c1-31-22-13-14-23(25(16-...,444.920013,Small molecule,100.00,ug.mL-1,'=',...,5.0,0.0,7.0,0.25,32.0,4.0,1.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
3,CHEMBL4089761,NC[C@H]1O[C@H](O[C@H]2[C@H](O)[C@@H](O[C@H]3O[...,C40H59FN10O14,VETOBWPFYDUOPP-FFKZLMLRSA-N,InChI=1S/C40H59FN10O14/c41-21-9-19-24(51(18-1-...,922.969971,Small molecule,1.50,ug.mL-1,'=',...,23.0,12.0,15.0,0.07,65.0,3.0,3.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
4,CHEMBL1241817,O=C(Cn1ncc2cc([N+](=O)[O-])ccc21)N/N=C/c1cccc(...,C16H12N6O5,DGJBCVGFPFQIOC-CAOOACKPSA-N,InChI=1S/C16H12N6O5/c23-16(19-17-8-11-2-1-3-13...,368.309998,Small molecule,16.00,ug.mL-1,'=',...,8.0,1.0,6.0,0.40,27.0,3.0,0.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34555,CHEMBL5574169,CC(C)(C)C(=O)Nc1nc(C(F)(F)F)c(-c2csc(Nc3cc(C(=...,C20H17F3N4O5S2,UCGIDVUDHVHHNH-UHFFFAOYSA-N,"InChI=1S/C20H17F3N4O5S2/c1-19(2,3)16(32)27-18-...",514.510010,NaN,32.00,ug.mL-1,'>',...,8.0,4.0,6.0,0.34,34.0,3.0,2.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
34556,CHEMBL5569723,CCC[C@@H]1Cc2cc3c(c(O)c2C(=O)O1)-c1c(c(OC)c2oc...,C46H54O20,JFJMNUYQKHCNOB-ZMBJTJKRSA-N,InChI=1S/C46H54O20/c1-6-7-19-10-18-11-20-30(38...,926.919983,NaN,1.00,ug.mL-1,'=',...,20.0,9.0,9.0,0.09,66.0,4.0,3.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
34557,CHEMBL5405424,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CCC...,C57H103N13O9,TWJPADFFDBMMLU-UILVTTEASA-N,InChI=1S/C57H103N13O9/c1-34(2)28-44(49(62)71)6...,1114.530029,NaN,4.00,ug.mL-1,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL359,Bacillus subtilis,Bacillus subtilis
34558,CHEMBL5590188,CCCCCC(=O)N/N=c1\sc2ccccc2n1C,C14H19N3OS,DSMZDRMEDMHOSJ-PEZBUJJGSA-N,InChI=1S/C14H19N3OS/c1-3-4-5-10-13(18)15-16-14...,277.390015,NaN,512.00,ug.mL-1,'>',...,4.0,1.0,5.0,0.66,19.0,2.0,0.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis


### Processing data files for `Bacillus_cereus` bacteria 

In [6]:
bacteriaName = 'Bacillus_cereus'  
bacteriaDataDir = os.path.join(dataDir, 'chEMBL/' + bacteriaName)
processBacteriaData(bacteriaName)


--- Processing Bioassay Data ---
Reading ChEMBL CSV file: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/chEMBL/Bacillus_cereus/Bacillus_cereus_bioactivities.csv
Dataset loaded successfully with Shape: (7696, 48)
Dataset shape after selecting columns: (7696, 48) → (7696, 12)
Cleaned bioassay data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Bacillus_cereusBioassay_chEMBL_cleaned.csv

--- Analyzing Standard Types ---
Standard types plot saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Bacillus_cereus_standard_types.png

--- Checking for NaN values ---
Smiles                   :     50 (  0.6%)
Standard Value           :  1,100 ( 14.3%)
Standard Units           :  1,194 ( 15.5%)
Standard Type            :      0 (  0.0%)
Standard Relation        :  1,098 ( 14.3%)
pChEMBL Value            :  7,687 ( 99.9%)
Molecule ChEMBL ID       :      0 (  0.0%)
Molecular Weight         :  

,molecule_chembl_id,smiles,molecular_formula,inchi_key,inchi,molecular_weight,type,standard_value,standard_units,standard_relation,...,hba,hbd,rotatable_bonds,qed_weighted,heavy_atoms,aromatic_rings,ro5_violations,Target ChEMBL ID,Target Name,Target Organism
0,CHEMBL372795,CN[C@@H]1[C@H](O[C@H]2[C@H](O[C@H]3[C@H](O)[C@...,C21H39N7O12,UCSJYZPVAKXKNQ-HZYVHMACSA-N,"InChI=1S/C21H39N7O12/c1-5-21(36,4-30)16(40-17-...",581.580017,Small molecule,20.530001,mm,'=',...,15.0,12.0,9.0,0.07,40.0,0.0,3.0,CHEMBL613070,Bacillus cereus,Bacillus cereus
1,CHEMBL2238351,O=c1oc2ccccc2cc1-c1nnc(Sc2nc(Oc3cccc4cccnc34)n...,C32H23N9O4S,FKBMYQGGGKPKOB-UHFFFAOYSA-N,InChI=1S/C32H23N9O4S/c42-28-22(19-21-7-1-2-10-...,629.659973,Small molecule,50.000000,ug.mL-1,'=',...,14.0,0.0,7.0,0.21,46.0,7.0,3.0,CHEMBL613070,Bacillus cereus,Bacillus cereus
2,CHEMBL4071700,CC1=CC2=C(CC(C)C(=O)C[C@H](C)O)C(=O)[C@](C)(O)...,C18H24O6,URYMFSUWSMOWTJ-UWFXQIJTSA-N,InChI=1S/C18H24O6/c1-9(15(20)6-10(2)19)5-13-12...,336.380005,Small molecule,NaN,NaN,NaN,...,6.0,3.0,5.0,0.70,24.0,0.0,0.0,CHEMBL613070,Bacillus cereus,Bacillus cereus
3,CHEMBL4077237,CO[C@@H](/C=C/C=C(\C)[C@H](O)[C@H](C)C(=O)O[C@...,C29H46O8,UWGTWPNAWMDWHM-AHGVLRMZSA-N,"InChI=1S/C29H46O8/c1-16-13-21-12-11-18(3)29(6,...",522.679993,Small molecule,NaN,NaN,NaN,...,8.0,4.0,11.0,0.18,37.0,0.0,1.0,CHEMBL613070,Bacillus cereus,Bacillus cereus
4,CHEMBL1276636,Cc1c(N2C(=O)C(Cl)=C(c3ccc[nH]3)C2=O)c(=O)n(-c2...,C19H15ClN4O3,PNPWLFUNSIHEEO-UHFFFAOYSA-N,InChI=1S/C19H15ClN4O3/c1-11-16(19(27)24(22(11)...,382.809998,Small molecule,NaN,NaN,NaN,...,5.0,1.0,3.0,0.71,27.0,3.0,0.0,CHEMBL613070,Bacillus cereus,Bacillus cereus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7691,CHEMBL5558032,Cn1cc(NC(=O)c2cc(NC(=O)c3ccc(/C=C/c4ccccc4)cc3...,C34H38N6O4,CSGUBISVSNYYLQ-MDZDMXLPSA-N,InChI=1S/C34H38N6O4/c1-38-24-29(21-30(38)33(42...,594.719971,NaN,70.000000,%,'=',...,7.0,3.0,11.0,0.18,44.0,4.0,1.0,CHEMBL613070,Bacillus cereus,Bacillus cereus
7692,CHEMBL5559579,CC(C)C[C@H](NC(=O)[C@H](CCCCN)NC(=O)CNC(=O)CNC...,C126H196N32O21,GTFVGDZKYFMXBQ-UIEXWCIESA-N,InChI=1S/C126H196N32O21/c1-72(2)53-97(118(171)...,2495.159912,NaN,32000.000000,nM,'>',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL613070,Bacillus cereus,Bacillus cereus
7693,CHEMBL5532125,CC(C)C[C@H](NC(=O)[C@H](CCCCN)NC(=O)CN)C(=O)N[...,C118H191N31O21,KMXQKIGASFXSSW-URGLQTMDSA-N,InChI=1S/C118H191N31O21/c1-67(2)50-89(110(162)...,2380.020020,NaN,4000.000000,nM,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL613070,Bacillus cereus,Bacillus cereus
7694,CHEMBL5556509,CC(C)C[C@H](NC(=O)[C@H](CCCCN)NC(=O)CNC(=O)CNC...,C120H195N31O21,MCZCEDIIGUEPSF-RLZKSDRCSA-N,InChI=1S/C120H195N31O21/c1-68(2)51-90(111(163)...,2408.080078,NaN,32000.000000,nM,'>',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL613070,Bacillus cereus,Bacillus cereus


### Processing data files for `Staphylococcus_aureus` bacteria 

In [7]:
bacteriaName = 'Staphylococcus_aureus'  
bacteriaDataDir = os.path.join(dataDir, 'chEMBL/' + bacteriaName)
processBacteriaData(bacteriaName)


--- Processing Bioassay Data ---
Reading ChEMBL CSV file: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/chEMBL/Staphylococcus_aureus/Staphylococcus_aureus_bioactivities.csv


/tmp/ipykernel_370924/4160898751.py:31: DtypeWarning: Columns (27,30,46) have mixed types. Specify dtype option on import or set low_memory=False.
  dataFrame = pd.read_csv(


Dataset loaded successfully with Shape: (232257, 48)
Dataset shape after selecting columns: (232257, 48) → (232257, 12)
Cleaned bioassay data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Staphylococcus_aureusBioassay_chEMBL_cleaned.csv

--- Analyzing Standard Types ---


/tmp/ipykernel_370924/4160898751.py:95: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Standard types plot saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Staphylococcus_aureus_standard_types.png

--- Checking for NaN values ---
Smiles                   :  1,272 (  0.5%)
Standard Value           : 20,780 (  8.9%)
Standard Units           : 29,172 ( 12.6%)
Standard Type            :      0 (  0.0%)
Standard Relation        : 20,772 (  8.9%)
pChEMBL Value            : 230,915 ( 99.4%)
Molecule ChEMBL ID       :      0 (  0.0%)
Molecular Weight         :  1,180 (  0.5%)
AlogP                    : 26,831 ( 11.6%)
Target ChEMBL ID         :      0 (  0.0%)
Target Name              :      0 (  0.0%)
Target Organism          :      0 (  0.0%)

Number of unique compounds for Staphylococcus_aureus: 83473

--- Processing Compounds Data ---
Successfully read compounds file with pandas
Compounds data shape: (83354, 29)
Cleaned compounds data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Sta

,molecule_chembl_id,smiles,molecular_formula,inchi_key,inchi,molecular_weight,type,standard_value,standard_units,standard_relation,...,hba,hbd,rotatable_bonds,qed_weighted,heavy_atoms,aromatic_rings,ro5_violations,Target ChEMBL ID,Target Name,Target Organism
0,CHEMBL507870,CCCCCCCCCCNCCN[C@@]1(C)C[C@H](O[C@H]2[C@H](Oc3...,C80H106Cl2N11O27P,ONUMZHGUFYIKPM-MXNFEBESSA-N,InChI=1S/C80H106Cl2N11O27P/c1-7-8-9-10-11-12-1...,1755.660034,Protein,3.300,NaN,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
1,CHEMBL1200628,CN[C@H](CC(C)C)C(=O)N[C@H]1C(=O)N[C@@H](CC(N)=...,C66H76Cl3N9O24,LCTORFDMHNKUSG-XTTLPDOESA-N,InChI=1S/C66H75Cl2N9O24.ClH/c1-23(2)12-34(71-5...,1485.729980,Small molecule,0.780,ug.mL-1,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
2,CHEMBL2164996,Cc1cc2c(cc1O)C1=C(C3=Cc4c(cc(O)c5cc(C)c(O)cc45...,C33H28O6,DFDVUOJRRAVNNF-UHFFFAOYSA-N,InChI=1S/C33H28O6/c1-14-7-18-16(10-24(14)34)17...,520.580017,Small molecule,25.000,ug.mL-1,'=',...,6.0,3.0,1.0,0.36,39.0,3.0,2.0,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
3,CHEMBL374478,CO[C@H]1/C=C/O[C@@]2(C)Oc3c(C)c(O)c4c(O)c(c(/C...,C43H58N4O12,JQXXHWHPUNPDRT-WLSIYKJHSA-N,InChI=1S/C43H58N4O12/c1-21-12-11-13-22(2)42(55...,822.950012,Small molecule,0.016,ug.mL-1,'=',...,15.0,6.0,4.0,0.11,59.0,2.0,3.0,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
4,CHEMBL376140,CN(C)c1cc(NC(=O)CNC(C)(C)C)c(O)c2c1C[C@H]1C[C@...,C29H39N5O8,FPZLLRFZJZRHSY-HJYUBDRYSA-N,"InChI=1S/C29H39N5O8/c1-28(2,3)31-11-17(35)32-1...",585.659973,Small molecule,0.125,ug.mL-1,'=',...,11.0,7.0,6.0,0.18,42.0,1.0,3.0,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232252,CHEMBL5570291,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CCC...,C60H100F3N13O10S,UKUVDXMWUHRYDL-GAQFJKPASA-N,InChI=1S/C60H100F3N13O10S/c1-37(2)30-48(53(68)...,1252.599976,NaN,32.000,ug.mL-1,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
232253,CHEMBL5574512,CC(C)C[C@H](N)CN(CC(=O)N[C@@H](CCCCN)C(=O)N[C@...,C57H102F3N13O10S,PCKQRAHXYIIWMC-KDXYNCTASA-N,InChI=1S/C57H102F3N13O10S/c1-34(2)27-40(64)32-...,1218.579956,NaN,32.000,ug.mL-1,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
232254,CHEMBL5405424,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CCC...,C57H103N13O9,TWJPADFFDBMMLU-UILVTTEASA-N,InChI=1S/C57H103N13O9/c1-34(2)28-44(49(62)71)6...,1114.530029,NaN,16.000,ug.mL-1,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
232255,CHEMBL5314354,NaN,NaN,NaN,NaN,NaN,Protein,16.000,ug.mL-1,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus


### Processing data files for `Enterococcus_faecalis` bacteria 

In [8]:
bacteriaName = 'Enterococcus_faecalis'  
bacteriaDataDir = os.path.join(dataDir, 'chEMBL/' + bacteriaName)
processBacteriaData(bacteriaName)


--- Processing Bioassay Data ---
Reading ChEMBL CSV file: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/chEMBL/Enterococcus_faecalis/Enterococcus_faecalis_bioactivities.csv
Dataset loaded successfully with Shape: (32387, 48)
Dataset shape after selecting columns: (32387, 48) → (32387, 12)
Cleaned bioassay data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Enterococcus_faecalisBioassay_chEMBL_cleaned.csv

--- Analyzing Standard Types ---


/tmp/ipykernel_370924/4160898751.py:95: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Standard types plot saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Enterococcus_faecalis_standard_types.png

--- Checking for NaN values ---
Smiles                   :    157 (  0.5%)
Standard Value           :  2,802 (  8.7%)
Standard Units           :  3,443 ( 10.6%)
Standard Type            :      0 (  0.0%)
Standard Relation        :  2,799 (  8.6%)
pChEMBL Value            : 32,336 ( 99.8%)
Molecule ChEMBL ID       :      0 (  0.0%)
Molecular Weight         :    157 (  0.5%)
AlogP                    :  5,076 ( 15.7%)
Target ChEMBL ID         :      0 (  0.0%)
Target Name              :      0 (  0.0%)
Target Organism          :      0 (  0.0%)

Number of unique compounds for Enterococcus_faecalis: 19447

--- Processing Compounds Data ---
Successfully read compounds file with pandas
Compounds data shape: (19428, 29)
Cleaned compounds data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Ente

,molecule_chembl_id,smiles,molecular_formula,inchi_key,inchi,molecular_weight,type,standard_value,standard_units,standard_relation,...,hba,hbd,rotatable_bonds,qed_weighted,heavy_atoms,aromatic_rings,ro5_violations,Target ChEMBL ID,Target Name,Target Organism
0,CHEMBL1090696,CCOC(=O)C1=C(O)CC(c2ccccc2)N(C(O)CN2CCOCC2)C1c...,C26H32N2O5,JSNKZNVIDARGEO-UHFFFAOYSA-N,InChI=1S/C26H32N2O5/c1-2-33-26(31)24-22(29)17-...,452.549988,Small molecule,128.0,ug.mL-1,'>',...,7.0,2.0,7.0,0.62,33.0,2.0,0.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
1,CHEMBL1093337,CCOC(=O)C1=C(O)CC(c2ccccc2)N(C(O)CSc2ccc3ccccc...,C32H31NO4S,HBEMNXKELWUETR-UHFFFAOYSA-N,InChI=1S/C32H31NO4S/c1-2-37-32(36)30-28(34)20-...,525.669983,Small molecule,128.0,ug.mL-1,'>',...,6.0,2.0,8.0,0.19,38.0,4.0,2.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
2,CHEMBL1091116,CCOC(=O)C1=C(O)CC(c2ccccc2)N(C(O)C(C)n2ccc3ccc...,C31H32N2O4,QGDCMLVOKPHNPM-UHFFFAOYSA-N,InChI=1S/C31H32N2O4/c1-3-37-31(36)28-27(34)20-...,496.609985,Small molecule,128.0,ug.mL-1,'>',...,6.0,2.0,7.0,0.30,37.0,4.0,1.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
3,CHEMBL4098498,CCCCCCCCCCCCCCCOP(=O)(CCN(CCCN)CCCN)OC[C@H]1O[...,C32H62N5O8P,WRBRSZMBGQHTLE-JNVRDQITSA-N,InChI=1S/C32H62N5O8P/c1-2-3-4-5-6-7-8-9-10-11-...,675.849976,Small molecule,50.0,ug.mL-1,'=',...,12.0,5.0,28.0,0.06,46.0,1.0,2.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
4,CHEMBL4069782,CCCCCCCCCCCCCCCOP(=O)(CCN(CCN)CCN)OC[C@H]1O[C@...,C30H58N5O8P,BFFZRGRWYYMLOO-WZICAWFSSA-N,InChI=1S/C30H58N5O8P/c1-2-3-4-5-6-7-8-9-10-11-...,647.799988,Small molecule,12.5,ug.mL-1,'=',...,12.0,5.0,26.0,0.07,44.0,1.0,2.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32382,CHEMBL5567548,O=C(O)c1cc(Nc2nc(-c3sc(NC(=O)c4ccccc4)nc3C(F)(...,C22H12F6N4O3S2,QQLBDHNRPSORQL-UHFFFAOYSA-N,"InChI=1S/C22H12F6N4O3S2/c23-21(24,25)12-6-11(1...",558.489990,NaN,32.0,ug ml-1,'>',...,7.0,3.0,6.0,0.22,37.0,4.0,2.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
32383,CHEMBL5560856,CCCCCCc1c(-c2ccc(OCC(=O)CN[C@H](CCCNC(=N)N)C(N...,C40H58N10O10,XCXOBKSLLXGVCD-LOYHVIPDSA-N,InChI=1S/C40H58N10O10/c1-3-4-5-6-9-28-35(54)34...,838.960022,NaN,64.0,ug.mL-1,'>',...,14.0,11.0,29.0,0.03,60.0,3.0,3.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
32384,CHEMBL5559737,CCCCCCNCCCCOc1ccc(-c2oc3cc(OC)cc(O)c3c(=O)c2CC...,C42H66N2O6,FLDHDGFIFIMEDE-UHFFFAOYSA-N,InChI=1S/C42H66N2O6/c1-5-8-11-14-21-36-41(46)4...,695.000000,NaN,64.0,ug.mL-1,'=',...,8.0,3.0,29.0,0.06,50.0,3.0,2.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
32385,CHEMBL5560242,CCCCCCc1c(-c2ccc(OCCCCN(C)C)cc2OCCCCN(C)C)oc2c...,C34H50N2O6,NSJASFJTZIBFCB-UHFFFAOYSA-N,InChI=1S/C34H50N2O6/c1-7-8-9-10-15-28-33(38)32...,582.780029,NaN,1.0,ug.mL-1,'=',...,8.0,1.0,19.0,0.16,42.0,3.0,2.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis


### Processing data files for `Listeria_monocytogenes` bacteria 

In [10]:
bacteriaName = 'Listeria_monocytogenes'  
bacteriaDataDir = os.path.join(dataDir, 'chEMBL/' + bacteriaName)
processBacteriaData(bacteriaName)


--- Processing Bioassay Data ---
Reading ChEMBL CSV file: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/chEMBL/Listeria_monocytogenes/Listeria_monocytogenes_bioactivities.csv
Dataset loaded successfully with Shape: (2701, 48)
Dataset shape after selecting columns: (2701, 48) → (2701, 12)
Cleaned bioassay data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Listeria_monocytogenesBioassay_chEMBL_cleaned.csv

--- Analyzing Standard Types ---
Standard types plot saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Listeria_monocytogenes_standard_types.png

--- Checking for NaN values ---
Smiles                   :     12 (  0.4%)
Standard Value           :    341 ( 12.6%)
Standard Units           :    405 ( 15.0%)
Standard Type            :      0 (  0.0%)
Standard Relation        :    341 ( 12.6%)
pChEMBL Value            :  2,685 ( 99.4%)
Molecule ChEMBL ID       :      0 (  0.0%)


,molecule_chembl_id,smiles,molecular_formula,inchi_key,inchi,molecular_weight,type,standard_value,standard_units,standard_relation,...,hba,hbd,rotatable_bonds,qed_weighted,heavy_atoms,aromatic_rings,ro5_violations,Target ChEMBL ID,Target Name,Target Organism
0,CHEMBL5315124,C[C@H]1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O)...,C36H42F2N6O9,SUIQUYDRLGGZOL-RCWTXCDDSA-N,InChI=1S/2C18H20FN3O4.H2O/c2*1-10-9-26-17-14-1...,740.760010,Small molecule,0.5,ug.mL-1,'=',...,6.0,1.0,2.0,0.87,26.0,2.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
1,CHEMBL1210162,O=C(O)c1ccc(/C=N/NC(=O)c2ccc(-c3nc4ccccc4[nH]3...,C22H16N4O3,HYEWSROTDZANKX-YDZHTSKRSA-N,InChI=1S/C22H16N4O3/c27-21(26-23-13-14-5-7-17(...,384.399994,Small molecule,400.0,ug.mL-1,'=',...,4.0,3.0,5.0,0.36,29.0,4.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
2,CHEMBL1236329,CC1=CC[C@@H]2C[C@H]1C2(C)C,C10H16,GRWFGVWFFZKLTI-RKDXNWHRSA-N,"InChI=1S/C10H16/c1-7-4-5-8-6-9(7)10(8,2)3/h4,8...",136.240005,Small molecule,0.0,%,'=',...,0.0,0.0,0.0,0.45,10.0,0.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
3,CHEMBL2269084,CCc1ccco1,C6H8O,HLPIHRDZBHXTFJ-UHFFFAOYSA-N,"InChI=1S/C6H8O/c1-2-6-4-3-5-7-6/h3-5H,2H2,1H3",96.129997,Small molecule,0.0,%,'=',...,1.0,0.0,1.0,0.52,7.0,1.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
4,CHEMBL545,CCO,C2H6O,LFQSCWFLJHTTHZ-UHFFFAOYSA-N,"InChI=1S/C2H6O/c1-2-3/h3H,2H2,1H3",46.070000,Small molecule,0.0,%,'=',...,1.0,1.0,0.0,0.41,3.0,0.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2696,CHEMBL4644199,FC(F)(F)c1ccc(NC(=S)N/N=C/c2ccc(Cl)cc2)cc1,C15H11ClF3N3S,BGKVILOXQJNMQS-AWQFTUOYSA-N,InChI=1S/C15H11ClF3N3S/c16-12-5-1-10(2-6-12)9-...,357.790009,Unknown,800.0,ug.mL-1,'=',...,2.0,2.0,3.0,0.48,23.0,2.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
2697,CHEMBL4632982,COc1ccc(/C=N/NC(=S)Nc2ccc(C(F)(F)F)cc2)cc1,C16H14F3N3OS,HPPOBRBATCMYHT-KEBDBYFISA-N,InChI=1S/C16H14F3N3OS/c1-23-14-8-2-11(3-9-14)1...,353.369995,Unknown,800.0,ug.mL-1,'=',...,3.0,2.0,4.0,0.49,24.0,2.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
2698,CHEMBL5171589,Clc1ccc2c(c1)C(NCCN1CCNCC1)=Nc1ccccc1O2,C19H21ClN4O,REXIYIBKBWXIKH-UHFFFAOYSA-N,InChI=1S/C19H21ClN4O/c20-14-5-6-17-15(13-14)19...,356.859985,NaN,1510.0,nM,'=',...,5.0,2.0,3.0,0.89,25.0,2.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
2699,CHEMBL5208663,FC(F)(F)Sc1ccc2c(c1)[C@H]1[C@@H]3CC[C@@H](C3)[...,C19H17F6N3S,BYXLRDJYOSEADO-KROWVVRQSA-N,"InChI=1S/C19H17F6N3S/c20-18(21,22)17-12(7-26-2...",433.420013,NaN,4.0,ug.mL-1,'=',...,3.0,2.0,2.0,0.43,29.0,2.0,1.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes


### Processing data files for `Bacillus_anthracis` bacteria 

In [11]:
bacteriaName = 'Bacillus_anthracis'  
bacteriaDataDir = os.path.join(dataDir, 'chEMBL/' + bacteriaName)
processBacteriaData(bacteriaName)


--- Processing Bioassay Data ---
Reading ChEMBL CSV file: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/chEMBL/Bacillus_anthracis/Bacillus_anthracis_bioactivities.csv
Dataset loaded successfully with Shape: (2964, 48)
Dataset shape after selecting columns: (2964, 48) → (2964, 12)
Cleaned bioassay data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Bacillus_anthracisBioassay_chEMBL_cleaned.csv

--- Analyzing Standard Types ---
Standard types plot saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Bacillus_anthracis_standard_types.png

--- Checking for NaN values ---
Smiles                   :      3 (  0.1%)
Standard Value           :    544 ( 18.4%)
Standard Units           :    917 ( 30.9%)
Standard Type            :      0 (  0.0%)
Standard Relation        :    544 ( 18.4%)
pChEMBL Value            :  2,922 ( 98.6%)
Molecule ChEMBL ID       :      0 (  0.0%)
Molecular Weight

,molecule_chembl_id,smiles,molecular_formula,inchi_key,inchi,molecular_weight,type,standard_value,standard_units,standard_relation,...,hba,hbd,rotatable_bonds,qed_weighted,heavy_atoms,aromatic_rings,ro5_violations,Target ChEMBL ID,Target Name,Target Organism
0,CHEMBL1200699,C[C@H]1c2cccc(O)c2C(=O)C2=C(O)[C@]3(O)C(=O)C(C...,C22H26N2O9,XQTWDDCIUJNLTR-CVHRZJFOSA-N,InChI=1S/C22H24N2O8.H2O/c1-7-8-5-4-6-9(25)11(8...,462.459991,Small molecule,6.63,NaN,'=',...,9.0,6.0,2.0,0.33,32.0,1.0,1.0,CHEMBL613904,Bacillus anthracis,Bacillus anthracis
1,CHEMBL1200699,C[C@H]1c2cccc(O)c2C(=O)C2=C(O)[C@]3(O)C(=O)C(C...,C22H26N2O9,XQTWDDCIUJNLTR-CVHRZJFOSA-N,InChI=1S/C22H24N2O8.H2O/c1-7-8-5-4-6-9(25)11(8...,462.459991,Small molecule,6.94,NaN,'=',...,9.0,6.0,2.0,0.33,32.0,1.0,1.0,CHEMBL613904,Bacillus anthracis,Bacillus anthracis
2,CHEMBL1200699,C[C@H]1c2cccc(O)c2C(=O)C2=C(O)[C@]3(O)C(=O)C(C...,C22H26N2O9,XQTWDDCIUJNLTR-CVHRZJFOSA-N,InChI=1S/C22H24N2O8.H2O/c1-7-8-5-4-6-9(25)11(8...,462.459991,Small molecule,7.14,NaN,'=',...,9.0,6.0,2.0,0.33,32.0,1.0,1.0,CHEMBL613904,Bacillus anthracis,Bacillus anthracis
3,CHEMBL1200699,C[C@H]1c2cccc(O)c2C(=O)C2=C(O)[C@]3(O)C(=O)C(C...,C22H26N2O9,XQTWDDCIUJNLTR-CVHRZJFOSA-N,InChI=1S/C22H24N2O8.H2O/c1-7-8-5-4-6-9(25)11(8...,462.459991,Small molecule,7.72,NaN,'=',...,9.0,6.0,2.0,0.33,32.0,1.0,1.0,CHEMBL613904,Bacillus anthracis,Bacillus anthracis
4,CHEMBL126,CC(=O)NC[C@H]1CN(c2ccc(N3CCOCC3)c(F)c2)C(=O)O1,C16H20FN3O4,TYZROVQLWOKYKF-ZDUSSCGKSA-N,InChI=1S/C16H20FN3O4/c1-11(21)18-9-13-10-20(16...,337.350006,Small molecule,NaN,NaN,NaN,...,5.0,1.0,4.0,0.89,24.0,1.0,0.0,CHEMBL613904,Bacillus anthracis,Bacillus anthracis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2959,CHEMBL1922604,CC(=O)N(O)CCCP(=O)([O-])O.[Na+],C5H11NNaO5P,LCFXFDGYDKNWMD-UHFFFAOYSA-M,"InChI=1S/C5H12NO5P.Na/c1-5(7)6(8)3-2-4-12(9,10...",219.110001,Small molecule,50.00,ug.mL-1,'=',...,3.0,3.0,4.0,0.33,12.0,0.0,0.0,CHEMBL613904,Bacillus anthracis,Bacillus anthracis
2960,CHEMBL1922611,CC(=O)N(O)CCCP(=O)(OC(C)OC(=O)OC(C)(C)C)OC(C)O...,C19H36NO11P,ALKUABCTJVZXGT-UHFFFAOYSA-N,InChI=1S/C19H36NO11P/c1-13(21)20(24)11-10-12-3...,485.470001,Small molecule,200.00,ug.mL-1,'=',...,11.0,1.0,10.0,0.15,32.0,0.0,1.0,CHEMBL613904,Bacillus anthracis,Bacillus anthracis
2961,CHEMBL5191499,O=C1CC(c2ccccc2)=NN1c1nc2ccccc2[nH]c1=O,C17H12N4O2,VDZREVWSAZOGEB-UHFFFAOYSA-N,InChI=1S/C17H12N4O2/c22-15-10-14(11-6-2-1-3-7-...,304.309998,NaN,7.80,ug.mL-1,'=',...,4.0,1.0,2.0,0.79,23.0,3.0,0.0,CHEMBL613904,Bacillus anthracis,Bacillus anthracis
2962,CHEMBL5206485,CC(C)c1ccc(C(=O)CCCN2CCC(O)(c3ccc(Cl)cc3)CC2)cc1,C24H30ClNO2,MTSDFMKGXLYXBW-UHFFFAOYSA-N,InChI=1S/C24H30ClNO2/c1-18(2)19-5-7-20(8-6-19)...,399.959991,NaN,NaN,%,NaN,...,3.0,1.0,7.0,0.63,28.0,2.0,1.0,CHEMBL613904,Bacillus anthracis,Bacillus anthracis


### Processing data files for `Yersinia_pestis` bacteria 

In [12]:
bacteriaName = 'Yersinia_pestis'  
bacteriaDataDir = os.path.join(dataDir, 'chEMBL/' + bacteriaName)
processBacteriaData(bacteriaName)


--- Processing Bioassay Data ---
Reading ChEMBL CSV file: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/chEMBL/Yersinia_pestis/Yersinia_pestis_bioactivities.csv
Dataset loaded successfully with Shape: (750, 48)
Dataset shape after selecting columns: (750, 48) → (750, 12)
Cleaned bioassay data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Yersinia_pestisBioassay_chEMBL_cleaned.csv

--- Analyzing Standard Types ---
Standard types plot saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Yersinia_pestis_standard_types.png

--- Checking for NaN values ---
Smiles                   :      3 (  0.4%)
Standard Value           :    140 ( 18.7%)
Standard Units           :    206 ( 27.5%)
Standard Type            :      0 (  0.0%)
Standard Relation        :    140 ( 18.7%)
pChEMBL Value            :    710 ( 94.7%)
Molecule ChEMBL ID       :      0 (  0.0%)
Molecular Weight         :     

,molecule_chembl_id,smiles,molecular_formula,inchi_key,inchi,molecular_weight,type,standard_value,standard_units,standard_relation,...,hba,hbd,rotatable_bonds,qed_weighted,heavy_atoms,aromatic_rings,ro5_violations,Target ChEMBL ID,Target Name,Target Organism
0,CHEMBL2368935,CC(=O)NCCCC(=O)N[C@@H](Cc1ccccc1)C(=O)N1Cc2ccc...,C139H198N26O20,SFPIVRLIPALSOI-HVVGCFLQSA-N,InChI=1S/C139H198N26O20/c1-89(166)147-72-34-62...,2553.270020,Unknown,196000.0,nM,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL614597,Yersinia pestis,Yersinia pestis
1,CHEMBL1255638,CC(=O)NCC(=O)N[C@@H](Cc1ccccc1)C(=O)N1Cc2ccccc...,C119H158N26O20,FNZKDKZHCIDOAU-OCDYNILSSA-N,InChI=1S/C119H158N26O20/c1-69(146)127-58-101(1...,2272.739990,Small molecule,220000.0,nM,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL614597,Yersinia pestis,Yersinia pestis
2,CHEMBL1255640,CC(=O)N[C@@H](CCCCN)C(=O)N[C@@H](CCCCN)C(=O)N[...,C131H182N26O20,RECKKAJQOHPRPH-ZHIDSBLVSA-N,InChI=1S/C131H182N26O20/c1-81(158)143-99(50-23...,2441.060059,Small molecule,13100.0,nM,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL614597,Yersinia pestis,Yersinia pestis
3,CHEMBL1255643,CC(C)C[C@H](NC(=O)[C@@H](N)CCCCN)C(=O)N1Cc2ccc...,C186H260N32O23,FAGFRRHJHKRXKQ-JLMPJPFUSA-N,InChI=1S/C186H260N32O23/c1-114(2)93-149(206-17...,3312.330078,Small molecule,150000.0,nM,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL614597,Yersinia pestis,Yersinia pestis
4,CHEMBL1200699,C[C@H]1c2cccc(O)c2C(=O)C2=C(O)[C@]3(O)C(=O)C(C...,C22H26N2O9,XQTWDDCIUJNLTR-CVHRZJFOSA-N,InChI=1S/C22H24N2O8.H2O/c1-7-8-5-4-6-9(25)11(8...,462.459991,Small molecule,560.0,nM,'=',...,9.0,6.0,2.0,0.33,32.0,1.0,1.0,CHEMBL614597,Yersinia pestis,Yersinia pestis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
745,CHEMBL1916860,O=S(=O)(c1ccccc1)N1N=C(c2ccccc2O)C[C@H]1c1ccc(...,C21H18N2O4S,OCMIHTWLNUIYFA-FQEVSTJZSA-N,InChI=1S/C21H18N2O4S/c24-16-12-10-15(11-13-16)...,394.450012,Small molecule,16000.0,nM,'>',...,5.0,2.0,4.0,0.71,28.0,3.0,0.0,CHEMBL614597,Yersinia pestis,Yersinia pestis
746,CHEMBL1916846,NC(=O)N1N=C(c2ccccc2O)C[C@H]1c1ccc(O)cc1,C16H15N3O3,UWJMRDJIYBMPLL-AWEZNQCLSA-N,InChI=1S/C16H15N3O3/c17-16(22)19-14(10-5-7-11(...,297.309998,Small molecule,31000.0,nM,'>',...,4.0,3.0,2.0,0.79,22.0,2.0,0.0,CHEMBL614597,Yersinia pestis,Yersinia pestis
747,CHEMBL1916842,Oc1ccc([C@@H]2CC(c3ccccc3O)=NN2)cc1,C15H14N2O2,RMOCLYAXEFWGCG-ZDUSSCGKSA-N,InChI=1S/C15H14N2O2/c18-11-7-5-10(6-8-11)13-9-...,254.289993,Small molecule,20000.0,nM,'=',...,4.0,3.0,2.0,0.77,19.0,2.0,0.0,CHEMBL614597,Yersinia pestis,Yersinia pestis
748,CHEMBL1916852,N=C(N)N1N=C(c2ccccc2O)C[C@H]1c1ccc(O)cc1,C16H16N4O2,FSZWTHSQGICTMA-AWEZNQCLSA-N,InChI=1S/C16H16N4O2/c17-16(18)20-14(10-5-7-11(...,296.329987,Small molecule,NaN,NaN,NaN,...,4.0,4.0,2.0,0.50,22.0,2.0,0.0,CHEMBL614597,Yersinia pestis,Yersinia pestis


### Combine the data set of all the bacteria into one large data set

In [13]:
# Explicitly list your CSV files
fileNames = [
    "Bacillus_subtilisData_chEMBL_combined.csv",
    "Bacillus_cereusData_chEMBL_combined.csv",
    "Staphylococcus_aureusData_chEMBL_combined.csv",
    "Enterococcus_faecalisData_chEMBL_combined.csv",
    "Listeria_monocytogenesData_chEMBL_combined.csv",
    "Bacillus_anthracisData_chEMBL_combined.csv",
    "Yersinia_pestisData_chEMBL_combined.csv"
]

# Read and concatenate
dfList = []
for fileName in fileNames:
    filePath = os.path.join(modelBuildingDataDir, fileName)
    df = pd.read_csv(filePath)
    df["Virus"] = fileName.replace("_Data_chEMBL_combined.csv", "")
    print(f"Loaded {fileName:35s} → Shape: {df.shape}")
    dfList.append(df)

# Concatenate all
allVirusData_chEMBL = pd.concat(dfList, ignore_index=True)
print(f"\nFinal combined shape: {allVirusData_chEMBL.shape}")

# Save
outFile = os.path.join(modelBuildingDataDir, "AllBacteriaData_chEMBL_combined.csv")
allVirusData_chEMBL.to_csv(outFile, index=False)
print(f"Saved merged CSV to: {outFile}")

Loaded Bacillus_subtilisData_chEMBL_combined.csv → Shape: (34560, 25)
Loaded Bacillus_cereusData_chEMBL_combined.csv → Shape: (7696, 25)
Loaded Staphylococcus_aureusData_chEMBL_combined.csv → Shape: (232257, 25)
Loaded Enterococcus_faecalisData_chEMBL_combined.csv → Shape: (32387, 25)
Loaded Listeria_monocytogenesData_chEMBL_combined.csv → Shape: (2701, 25)
Loaded Bacillus_anthracisData_chEMBL_combined.csv → Shape: (2964, 25)
Loaded Yersinia_pestisData_chEMBL_combined.csv → Shape: (750, 25)

Final combined shape: (313315, 25)
Saved merged CSV to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/AllBacteriaData_chEMBL_combined.csv
